# Drug RAG — 전체 임베딩 파이프라인

**실행 전 체크리스트**
- [ ] 런타임 유형 변경 → T4 GPU
- [ ] Secrets 등록: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_DEFAULT_REGION, S3_BUCKET

**모드 선택**
- `FULL_REFRESH = True` : 전체 데이터 재임베딩 (DB 초기화 후 전체 적재)
- `FULL_REFRESH = False`: symptom만 재임베딩 (기존 symptom 삭제 후 재적재)

In [1]:
# Cell 1: 모드 설정
FULL_REFRESH = False  # True: 전체 재임베딩 / False: symptom만 재임베딩
print(f'모드: {"전체 재임베딩" if FULL_REFRESH else "symptom 재임베딩"}')

모드: symptom 재임베딩


In [14]:
# Cell 2: PostgreSQL + pgvector 설치
!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib
!apt-get -qq install -y postgresql-server-dev-14
!cd /tmp/pgvector && make clean -s && make -s && sudo make install -s
!ls /usr/share/postgresql/14/extension/vector.control 2>/dev/null && echo "설치 완료" || echo "실패"
import subprocess
r = subprocess.run(['sudo', '-u', 'postgres', 'psql', '-c', 'CREATE EXTENSION IF NOT EXISTS vector;'], capture_output=True, text=True)
if r.returncode != 0:
    print('pgvector 소스 빌드 중...')
    !cd /tmp && git clone --branch v0.7.0 https://github.com/pgvector/pgvector.git 2>/dev/null
    !cd /tmp/pgvector && make -s && make install -s
print('PostgreSQL 설치 완료')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
(Reading database ... 124550 files and directories currently installed.)
Preparing to unpack .../libc6-dev_2.35-0ubuntu3.14_amd64.deb ...
Unpacking libc6-dev:amd64 (2.35-0ubuntu3.14) over (2.35-0ubuntu3.9) ...
Preparing to unpack .../libc-dev-bin_2.35-0ubuntu3.14_amd64.deb ...
Unpacking libc-dev-bin (2.35-0ubuntu3.14) over (2.35-0ubuntu3.9) ...
Preparing to unpack .../libc6_2.35-0ubuntu3.14_amd64.deb ...
Unpacking libc6:amd64 (2.35-0ubuntu3.14) over (2.35-0ubuntu3.9) ...
Setting up libc6:amd64 (2.35-0ubuntu3.14) ...
Selecting previously unselected package python3-yaml.
(Reading database ... 124550 files and directories currently installed.)
Preparing to unpack .../00-python3-yaml_5.4.1-1ubuntu1_amd64.deb ...
Unpacking python3-yaml (5.4.1-1ubuntu1) ...
Selecting previously 

In [16]:
# Cell 3: PostgreSQL 시작 + DB 생성
import time
!service postgresql start
time.sleep(3)
!sudo -u postgres psql -c "CREATE DATABASE drug_rag;"
!sudo -u postgres psql -d drug_rag -c "CREATE EXTENSION IF NOT EXISTS vector;"
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'drugrag123';"
print('PostgreSQL 준비 완료')

 * Starting PostgreSQL 14 database server
   ...done.
ERROR:  database "drug_rag" already exists
CREATE EXTENSION
ALTER ROLE
PostgreSQL 준비 완료


In [4]:
# Cell 4: Python 패키지 설치
!pip install -q boto3 sentence-transformers psycopg2-binary pgvector langchain-text-splitters tqdm
print('패키지 설치 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.8 MB/s eta 0:00:00
패키지 설치 완료


In [5]:
# Cell 5: 환경변수 로드
import os
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = userdata.get('AWS_DEFAULT_REGION')
S3_BUCKET = userdata.get('S3_BUCKET')

PG_HOST = 'localhost'
PG_PORT = '5432'
PG_DB   = 'drug_rag'
PG_USER = 'postgres'
PG_PASS = 'drugrag123'

print(f'S3 버킷: {S3_BUCKET}')

S3 버킷: drug-rag-datalake-331145994962


In [6]:
# Cell 6: GPU 확인
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU 없음. 런타임 유형을 T4 GPU로 변경하세요.')

디바이스: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [7]:
# Cell 7: 임베딩 모델 로드
from sentence_transformers import SentenceTransformer
MODEL_NAME = 'intfloat/multilingual-e5-large'
print(f'로딩 중: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME, device=device)
EMBED_DIM = model.get_sentence_embedding_dimension()
print(f'완료 / 차원: {EMBED_DIM}')

로딩 중: intfloat/multilingual-e5-large


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

완료 / 차원: 1024


/tmp/ipykernel_612/1128364432.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = model.get_sentence_embedding_dimension()


In [17]:
# Cell 8: S3 유틸 + PostgreSQL 연결
import json, boto3, psycopg2
from psycopg2.extras import execute_values

s3 = boto3.client('s3', region_name=os.environ['AWS_DEFAULT_REGION'])

def list_s3_keys(prefix):
    paginator = s3.get_paginator('list_objects_v2')
    keys = []
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        for obj in page.get('Contents', []):
            if obj['Key'].endswith('.json'):
                keys.append(obj['Key'])
    return keys

def download_json(key):
    res = s3.get_object(Bucket=S3_BUCKET, Key=key)
    return json.loads(res['Body'].read().decode('utf-8'))

conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS)
conn.autocommit = False
print('PostgreSQL 연결 완료')

PostgreSQL 연결 완료


In [18]:
# pgvector 확장 적용
with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
conn.commit()
print('pgvector 확장 완료')

pgvector 확장 완료


In [20]:
# Cell 9: 테이블 생성
with conn.cursor() as cur:
    cur.execute(f'''
        CREATE TABLE IF NOT EXISTS drug_documents (
            id           SERIAL PRIMARY KEY,
            doc_id       TEXT UNIQUE NOT NULL,
            doc_type     TEXT NOT NULL,
            chunk_index  INTEGER DEFAULT 0,
            text         TEXT NOT NULL,
            embedding    vector({EMBED_DIM}),
            name         TEXT,
            source_name  TEXT,
            source_url   TEXT,
            license      TEXT,
            provider     TEXT,
            item_url     TEXT,
            manufacturer TEXT,
            image_url    TEXT,
            dur_type     TEXT,
            ingredient_a TEXT,
            ingredient_b TEXT,
            metadata     JSONB,
            created_at   TIMESTAMP DEFAULT NOW()
        );
    ''')
    cur.execute('CREATE INDEX IF NOT EXISTS drug_docs_type_idx ON drug_documents (doc_type);')
conn.commit()
print('테이블 생성 완료')

테이블 생성 완료


In [21]:
# Cell 10: 청킹 함수
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_CONFIG = {
    'drug_efficacy':    {'size': 512, 'overlap': 64},
    'drug_usage':       {'size': 256, 'overlap': 32},
    'drug_caution':     {'size': 384, 'overlap': 64},
    'drug_side_effect': {'size': 384, 'overlap': 64},
    'dur_interaction':  {'size': 256, 'overlap': 0},
    'symptom_info':     {'size': 512, 'overlap': 64},
    'symptom_redflag':  {'size': 256, 'overlap': 32},
}

def chunk_document(doc):
    doc_type = doc.get('doc_type', 'drug_efficacy')
    cfg = CHUNK_CONFIG.get(doc_type, {'size': 512, 'overlap': 64})
    text = doc.get('text', '')
    if not text:
        return []
    if doc_type in ('dur_interaction', 'symptom_redflag') or len(text) <= cfg['size']:
        return [{'chunk_index': 0, 'text': text}]
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg['size'], chunk_overlap=cfg['overlap'], separators=['|', ' ', '']
    )
    return [{'chunk_index': i, 'text': c} for i, c in enumerate(splitter.split_text(text))]

print('청킹 함수 준비 완료')

청킹 함수 준비 완료


In [22]:
# Cell 11: 임베딩 + 적재 함수
import numpy as np
from tqdm import tqdm

def embed_texts(texts, batch_size=64):
    passages = [f'passage: {t}' for t in texts]
    return model.encode(passages, batch_size=batch_size, show_progress_bar=False, normalize_embeddings=True)

def upsert_chunks(docs, batch_size=64):
    rows = []
    seen_doc_ids = set()
    for doc in docs:
        chunks = chunk_document(doc)
        meta = doc.get('metadata', {})
        for chunk in chunks:
            doc_id = f"{doc['doc_id']}_c{chunk['chunk_index']}"
            if doc_id in seen_doc_ids:
                continue
            seen_doc_ids.add(doc_id)
            rows.append({
                'doc_id':       doc_id,
                'doc_type':     doc.get('doc_type', ''),
                'chunk_index':  chunk['chunk_index'],
                'text':         chunk['text'],
                'name':         meta.get('name', ''),
                'source_name':  meta.get('source_name', ''),
                'source_url':   meta.get('source_url', ''),
                'license':      meta.get('license', ''),
                'provider':     meta.get('provider', ''),
                'item_url':     meta.get('item_url', ''),
                'manufacturer': meta.get('manufacturer', ''),
                'image_url':    meta.get('image_url', ''),
                'dur_type':     meta.get('dur_type', ''),
                'ingredient_a': meta.get('ingredient_a_kor', '') or meta.get('ingredient_a', ''),
                'ingredient_b': meta.get('ingredient_b_kor', '') or meta.get('ingredient_b', ''),
                'metadata':     json.dumps(meta, ensure_ascii=False),
            })
    if not rows:
        return 0

    # 기존 doc_id 스킵
    doc_ids = [r['doc_id'] for r in rows]
    with conn.cursor() as cur:
        cur.execute('SELECT doc_id FROM drug_documents WHERE doc_id = ANY(%s)', (doc_ids,))
        existing = {r[0] for r in cur.fetchall()}
    rows = [r for r in rows if r['doc_id'] not in existing]
    if not rows:
        return 0

    embeddings = embed_texts([r['text'] for r in rows], batch_size=batch_size)
    with conn.cursor() as cur:
        execute_values(cur, '''
            INSERT INTO drug_documents (
                doc_id, doc_type, chunk_index, text, embedding,
                name, source_name, source_url, license, provider,
                item_url, manufacturer, image_url,
                dur_type, ingredient_a, ingredient_b, metadata
            ) VALUES %s
            ON CONFLICT (doc_id) DO NOTHING
        ''', [
            (r['doc_id'], r['doc_type'], r['chunk_index'], r['text'],
             embeddings[i].tolist(),
             r['name'], r['source_name'], r['source_url'], r['license'], r['provider'],
             r['item_url'], r['manufacturer'], r['image_url'],
             r['dur_type'], r['ingredient_a'], r['ingredient_b'], r['metadata'])
            for i, r in enumerate(rows)
        ])
    conn.commit()
    return len(rows)

print('임베딩/적재 함수 준비 완료')

임베딩/적재 함수 준비 완료


In [23]:
# Cell 12: Silver 처리 (FULL_REFRESH 모드에 따라 분기)
ALL_PREFIXES = [
    'silver/refined/drugs/',
    'silver/refined/dur/combination_ban/',
    'silver/refined/dur/pregnancy_ban/',
    'silver/refined/dur/elderly_caution/',
    'silver/refined/dur/dose_caution/',
    'silver/refined/dur/effect_duplicate/',
    'silver/refined/dur/age_specific_ban/',
    'silver/refined/dur/dosing_period_caution/',
    'silver/refined/symptoms/',
]

SYMPTOM_ONLY_PREFIXES = [
    'silver/refined/symptoms/',
]

SILVER_PREFIXES = ALL_PREFIXES if FULL_REFRESH else SYMPTOM_ONLY_PREFIXES

# symptom만 재임베딩 시 기존 symptom 삭제
if not FULL_REFRESH:
    print('기존 symptom 데이터 삭제 중...')
    with conn.cursor() as cur:
        cur.execute("DELETE FROM drug_documents WHERE doc_type IN ('symptom_info', 'symptom_redflag');")
    conn.commit()
    print('기존 symptom 삭제 완료')
else:
    print('전체 재임베딩 모드: DB 초기화')
    with conn.cursor() as cur:
        cur.execute('TRUNCATE TABLE drug_documents;')
    conn.commit()
    print('DB 초기화 완료')

total_saved = 0
for prefix in SILVER_PREFIXES:
    keys = list_s3_keys(prefix)
    if not keys:
        print(f'스킵: {prefix}')
        continue
    label = prefix.rstrip('/').split('/')[-1]
    print(f'\n▶ {label} ({len(keys)}개 파일)')
    for key in tqdm(keys, desc=label):
        try:
            data = download_json(key)
            items = data.get('items', [])
            for i in range(0, len(items), 200):
                try:
                    total_saved += upsert_chunks(items[i:i+200])
                except Exception as e:
                    conn.rollback()
                    print(f'  오류: {str(e)[:60]}')
        except Exception as e:
            print(f'오류 {key}: {str(e)[:60]}')

print(f'\n전체 완료: {total_saved:,}개 청크 적재')

기존 symptom 데이터 삭제 중...
기존 symptom 삭제 완료

▶ symptoms (5개 파일)


symptoms: 100%|██████████| 5/5 [02:03<00:00, 24.67s/it]


전체 완료: 2,125개 청크 적재


In [24]:
# Cell 13: 적재 결과 확인
with conn.cursor() as cur:
    cur.execute('SELECT doc_type, COUNT(*) FROM drug_documents GROUP BY doc_type ORDER BY COUNT(*) DESC')
    rows = cur.fetchall()
total = 0
for doc_type, cnt in rows:
    print(f'  {doc_type:<25} {cnt:>8,}건')
    total += cnt
print(f'  합계{"":21} {total:>8,}건')

  symptom_info                 1,804건
  symptom_redflag                321건
  합계                         2,125건


In [25]:
# Cell 14: 벡터 인덱스 생성 (적재 완료 후)
print('벡터 인덱스 생성 중...')
with conn.cursor() as cur:
    cur.execute('''
        CREATE INDEX IF NOT EXISTS drug_docs_embedding_idx
        ON drug_documents USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100);
    ''')
conn.commit()
print('벡터 인덱스 생성 완료')

벡터 인덱스 생성 중...
벡터 인덱스 생성 완료


In [26]:
# Cell 15: 검색 테스트
def search(query, doc_type=None, top_k=5):
    q_emb = model.encode(f'query: {query}', normalize_embeddings=True).tolist()
    fetch_k = top_k * 10
    with conn.cursor() as cur:
        if doc_type:
            cur.execute('''
                SELECT name, source_name, item_url, text, metadata,
                       1-(embedding <=> %s::vector) AS score
                FROM drug_documents WHERE doc_type=%s
                ORDER BY embedding <=> %s::vector LIMIT %s
            ''', (q_emb, doc_type, q_emb, fetch_k))
        else:
            cur.execute('''
                SELECT name, source_name, item_url, text, metadata,
                       1-(embedding <=> %s::vector) AS score
                FROM drug_documents
                ORDER BY embedding <=> %s::vector LIMIT %s
            ''', (q_emb, q_emb, fetch_k))
        rows = cur.fetchall()

    seen = set()
    results = []
    for name, source, url, text, meta, score in rows:
        meta_dict = json.loads(meta) if isinstance(meta, str) else (meta or {})
        dedup_key = meta_dict.get('parent_doc_id') or meta_dict.get('item_url') or name
        if dedup_key not in seen:
            seen.add(dedup_key)
            results.append((name, source, text, score))
        if len(results) >= top_k:
            break
    return results

print('=== 두통에 먹는 약 ===')
for name, source, text, score in search('두통에 먹는 약', doc_type='drug_efficacy'):
    print(f'[{score:.3f}] {name} ({source})')
    print(f'  {text[:100]}')

print('\n=== 이부프로펜 병용금기 ===')
for name, source, text, score in search('이부프로펜 병용금기', doc_type='dur_interaction'):
    print(f'[{score:.3f}] {text[:120]}')

print('\n=== 응급 위험 증상 ===')
for name, source, text, score in search('즉시 병원 가야 하는 증상', doc_type='symptom_redflag'):
    print(f'[{score:.3f}] {name} ({source})')
    print(f'  {text[:100]}')

=== 두통에 먹는 약 ===

=== 이부프로펜 병용금기 ===

=== 응급 위험 증상 ===
[0.826] 항문직장농양과 치루(항문직장농양) (국가건강정보포털)
  [치료] 항문직장농양이 진단되면 즉시 절개하여 배농 시켜 주어야 합니다. 때로 수술하지 않고 항생제만 투여하여 염증이 가라앉는 수도 있지만, 주위로 염증이 더 파급되면서 심해지는 
[0.820] 일광화상 (국가건강정보포털)
  [자주하는 질문] Q. 일광 화상 후 허물이 벗겨지는데 어떻게 해야 합니까? A. 허물이 벗겨지는 증상은 정상적인 회복 과정 중 인설이 한꺼번에 벗겨지면서 나타나는 현상이므로, 억
[0.816] 외이도염(알레르기외이도염) (국가건강정보포털)
  [치료] 치료를 위해서 원인 물질을 찾아내어 회피하는 것이 가장 중요합니다. 먼저 알레르기에 의한 것을 의심하는 것이 필요하며, 의심된다면 첩포검사 등을 통하여 확진하고 원인 물질


In [27]:
# Cell 16: pg_dump → S3 저장 (세션 종료 전 필수!)
import subprocess
from datetime import datetime

dump_path = '/tmp/drug_rag_pgvector.dump'
s3_key = f'gold/pgvector_dump/{datetime.now().strftime("%Y-%m-%d")}/drug_rag_pgvector.dump'

print('pg_dump 실행 중...')
r = subprocess.run(
    ['sudo', '-u', 'postgres', 'pg_dump', '-Fc', '-d', 'drug_rag', '-f', dump_path],
    capture_output=True, text=True
)
if r.returncode == 0:
    size_mb = os.path.getsize(dump_path) / 1024 / 1024
    print(f'dump 크기: {size_mb:.1f} MB')
    s3.upload_file(dump_path, S3_BUCKET, s3_key)
    print(f'S3 저장 완료: s3://{S3_BUCKET}/{s3_key}')
else:
    print(f'오류: {r.stderr}')

pg_dump 실행 중...
dump 크기: 11.7 MB
S3 저장 완료: s3://drug-rag-datalake-331145994962/gold/pgvector_dump/2026-09-07/drug_rag_pgvector.dump


## Cell 17: 로컬 restore 명령어

```cmd
aws s3 cp s3://drug-rag-datalake-331145994962/gold/pgvector_dump/YYYY-MM-DD/drug_rag_pgvector.dump D:\...\drug_rag_pgvector.dump

docker exec -it drug_rag_postgres psql -U airflow -c "SELECT pg_terminate_backend(pid) FROM pg_stat_activity WHERE datname = 'drug_rag' AND pid <> pg_backend_pid();"
docker exec -it drug_rag_postgres psql -U airflow -c "DROP DATABASE IF EXISTS drug_rag;"
docker exec -it drug_rag_postgres psql -U airflow -c "CREATE DATABASE drug_rag;"
docker exec -it drug_rag_postgres psql -U airflow -d drug_rag -c "CREATE EXTENSION IF NOT EXISTS vector;"
docker cp D:\...\drug_rag_pgvector.dump drug_rag_postgres:/tmp/
docker exec -it drug_rag_postgres pg_restore -U airflow -d drug_rag /tmp/drug_rag_pgvector.dump

docker exec -it drug_rag_postgres psql -U airflow -d drug_rag -c "SELECT doc_type, COUNT(*) FROM drug_documents GROUP BY doc_type ORDER BY COUNT(*) DESC;"
```